# Bangla License Plate OCR Workflow

This notebook is the OCR stage for the current clean-dataset pipeline. It reads plate crops created by `deblurgan_workflow.ipynb` from `clean_dataset_processed`, creates DeblurGAN-restored OCR inputs, and runs OCR prediction code on those images.

Important: the current `clean dataset` only provides full-plate YOLO boxes, not character-level OCR text labels. Because of that, this notebook does not read the old `dataset/` folder and cannot compute supervised OCR accuracy/CER/WER unless clean-dataset text labels are added later.


In [ ]:
# Optional one-time installs if your notebook environment is missing OCR packages.
# %pip install easyocr pytesseract tqdm pandas matplotlib opencv-python pillow

# Tesseract also needs the system application plus Bengali traineddata.
# Windows example after installing Tesseract OCR:
# import pytesseract
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"



In [ ]:
from pathlib import Path
from collections import Counter
import ast
import json
import math
import random
import re
import unicodedata
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

try:
    from PIL import Image, ImageDraw, ImageFont
except Exception:
    Image = ImageDraw = ImageFont = None

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
PROJECT_ROOT = Path.cwd()
DEBLUR_SOURCE_DATASET = PROJECT_ROOT / "clean_dataset_processed"
PAIR_ROOT = PROJECT_ROOT / "deblurgan_data" / "clean_dataset_paired_plate_crops_psnr_v2"
DEBLUR_NOTEBOOK = PROJECT_ROOT / "deblurgan_workflow.ipynb"
DEBLUR_CHECKPOINT = PROJECT_ROOT / "runs" / "deblurgan_clean_dataset_irv2_fpn_psnr_v2" / "checkpoints" / "best_generator.pt"

OCR_RUN_ROOT = PROJECT_ROOT / "runs" / "ocr_clean_dataset_psnr_v2"
MANIFEST_ROOT = OCR_RUN_ROOT / "manifests"
DEBLURRED_OCR_INPUT_ROOT = OCR_RUN_ROOT / "deblurred_ocr_inputs"
PREDICTION_ROOT = OCR_RUN_ROOT / "predictions"
FIGURE_ROOT = OCR_RUN_ROOT / "figures"
CRNN_CHECKPOINT_ROOT = OCR_RUN_ROOT / "checkpoints"
STRUCTURED_CHECKPOINT_ROOT = CRNN_CHECKPOINT_ROOT / "structured_ocr"
TOP_STRUCTURED_CHECKPOINT = STRUCTURED_CHECKPOINT_ROOT / "top_component_classifier_best.pt"
BOTTOM_FIXED_DIGIT_CHECKPOINT = STRUCTURED_CHECKPOINT_ROOT / "bottom_fixed_digit_best.pt"

for folder in [MANIFEST_ROOT, DEBLURRED_OCR_INPUT_ROOT, PREDICTION_ROOT, FIGURE_ROOT, CRNN_CHECKPOINT_ROOT, STRUCTURED_CHECKPOINT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "valid", "test"]
PLATE_HEIGHT = 96
PLATE_WIDTH = 256
DEBLUR_BATCH_SIZE = 1
OVERWRITE_DEBLURRED_OCR_INPUTS = False
HAS_OCR_GROUND_TRUTH = False

# OCR input variants. "sharp" is an oracle comparison from paired data, not an input available in deployment.
OCR_IMAGE_VARIANTS = {
    "blurry": "blurry_crop",
    "deblurred": "ocr_image",
    "sharp": "sharp_crop",
}
OCR_EVALUATION_VARIANTS = ["blurry", "deblurred", "sharp"]
DEPLOYABLE_OCR_VARIANTS = ["blurry", "deblurred"]
DEFAULT_DEPLOYABLE_OCR_VARIANT = "deblurred"

# The clean dataset has plate-box labels, not OCR text labels. Structured OCR training is disabled here.
OCR_IMAGE_HEIGHT = 48
OCR_IMAGE_WIDTH = 288
BOTTOM_DIGIT_COUNT = 6
STRUCTURED_BATCH_SIZE = 32
STRUCTURED_EPOCHS = 40
STRUCTURED_LEARNING_RATE = 5e-4
STRUCTURED_WEIGHT_DECAY = 1e-4
RUN_STRUCTURED_OCR_TRAINING = False
RETRAIN_STRUCTURED_OCR = False
USE_OCR_AUGMENTATION = True
USE_CLASS_BALANCING = True
USE_SYNTHETIC_OCR_DATA = False
SYNTHETIC_SAMPLES_PER_TOP_CLASS = 60
SYNTHETIC_RANDOM_SAMPLES = 350
SYNTHETIC_RARITY_BOOST_CAP = 4
TOP_CLASS_LOSS_WEIGHT = 1.5
TOP_METRO_LOSS_WEIGHT = 0.5

EASYOCR_LANGS = ["bn", "en"]
TESSERACT_LANG = "ben"
TESSERACT_CONFIG = "--oem 1 --psm 7"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEBLUR_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Deblur source dataset:", DEBLUR_SOURCE_DATASET)
print("Deblur pair root:", PAIR_ROOT)
print("Deblur checkpoint:", DEBLUR_CHECKPOINT)
print("OCR run root:", OCR_RUN_ROOT)
print("OCR ground truth available:", HAS_OCR_GROUND_TRUTH)
print("Torch device:", DEVICE)
print("Deblur inference device:", DEBLUR_DEVICE)


In [ ]:
def project_path(value):
    text = str(value).replace("\\", "/")
    path = Path(text)
    return path if path.is_absolute() else PROJECT_ROOT / path


def relative_text(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def require_path(path, label):
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {label}: {path}")


require_path(DEBLUR_SOURCE_DATASET, "clean processed dataset")
require_path(PAIR_ROOT, "clean-dataset DeblurGAN paired crop root")
require_path(DEBLUR_NOTEBOOK, "DeblurGAN workflow notebook")
require_path(DEBLUR_CHECKPOINT, "DeblurGAN generator checkpoint")


## 1. Build OCR Input Manifest

The current pipeline starts from `clean_dataset_processed` and the DeblurGAN paired crop manifest in `deblurgan_data/clean_dataset_paired_plate_crops_psnr_v2`. This section builds OCR input manifests from those clean-dataset plate crops only. No files are read from the old `dataset/` folder.


In [ ]:
BANGLA_DIGITS = {
    "0": "\u09e6", "1": "\u09e7", "2": "\u09e8", "3": "\u09e9", "4": "\u09ea",
    "5": "\u09eb", "6": "\u09ec", "7": "\u09ed", "8": "\u09ee", "9": "\u09ef",
}

TARGET_TOKEN_MAP = {
    **BANGLA_DIGITS,
    "dhaka": "\u09a2\u09be\u0995\u09be",
    "sylhet": "\u09b8\u09bf\u09b2\u09c7\u099f",
    "chattro": "\u099a\u099f\u09cd\u099f",
    "metro": "\u09ae\u09c7\u099f\u09cd\u09b0\u09cb",
    "ka": "\u0995",
    "kha": "\u0996",
    "ga": "\u0997",
    "gha": "\u0998",
    "cha": "\u099a",
    "chha": "\u099b",
    "ja": "\u099c",
    "jha": "\u099d",
    "Ta": "\u099f",
    "Tha": "\u09a0",
    "ba": "\u09ac",
    "bha": "\u09ad",
    "m": "\u09ae",
    "na": "\u09a8",
    "sa": "\u09b8",
}

BANGLA_DIGIT_VALUES = set(BANGLA_DIGITS.values())
FULL_PLATE_CLASS = "license-plate-number"


def load_yolo_class_names(data_yaml_path):
    text = Path(data_yaml_path).read_text(encoding="utf-8")
    inline_match = re.search(r"names:\s*(\[[\s\S]*?\])", text)
    if inline_match:
        return list(ast.literal_eval(inline_match.group(1)))

    names = {}
    in_names = False
    for line in text.splitlines():
        if line.strip().startswith("names:"):
            in_names = True
            continue
        if in_names:
            if not line.startswith(" ") and not line.startswith("\t"):
                break
            match = re.match(r"\s*(\d+)\s*:\s*(.+?)\s*$", line)
            if match:
                names[int(match.group(1))] = match.group(2).strip().strip("'\"")
    if names:
        return [names[index] for index in sorted(names)]
    raise ValueError(f"Could not parse class names from {data_yaml_path}")


CLASS_NAMES = load_yolo_class_names(DEBLUR_SOURCE_DATASET / "data.yaml")
print("Clean dataset class count:", len(CLASS_NAMES))
print("OCR token classes kept for normalization/synthetic helpers:", sorted(TARGET_TOKEN_MAP))
print("No OCR character labels are read from the old dataset folder.")


def read_character_tokens(label_path, class_names=CLASS_NAMES):
    tokens = []
    label_path = Path(label_path)
    if not label_path.exists():
        return tokens
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            class_id = int(float(parts[0]))
            x_center, y_center, width, height = map(float, parts[1:5])
        except ValueError:
            continue
        if class_id < 0 or class_id >= len(class_names):
            continue
        class_name = str(class_names[class_id]).strip()
        if class_name == FULL_PLATE_CLASS or class_name not in TARGET_TOKEN_MAP:
            continue
        tokens.append({
            "token": TARGET_TOKEN_MAP[class_name],
            "class_name": class_name,
            "x_center": x_center,
            "y_center": y_center,
            "width": width,
            "height": height,
        })
    return tokens


def split_token_rows(tokens):
    if not tokens:
        return []
    median_height = float(np.median([item["height"] for item in tokens]))
    row_threshold = max(0.10, median_height * 0.75)
    rows = []
    for item in sorted(tokens, key=lambda value: value["y_center"]):
        best_index = None
        best_distance = None
        for index, row in enumerate(rows):
            row_y = float(np.mean([entry["y_center"] for entry in row]))
            distance = abs(item["y_center"] - row_y)
            if distance <= row_threshold and (best_distance is None or distance < best_distance):
                best_index = index
                best_distance = distance
        if best_index is None:
            rows.append([item])
        else:
            rows[best_index].append(item)
    rows.sort(key=lambda row: float(np.mean([entry["y_center"] for entry in row])))
    for row in rows:
        row.sort(key=lambda value: value["x_center"])
    return rows


def join_row_tokens(tokens):
    pieces = []
    digit_buffer = ""
    for token in tokens:
        if all(char in BANGLA_DIGIT_VALUES for char in token):
            digit_buffer += token
        else:
            if digit_buffer:
                pieces.append(digit_buffer)
                digit_buffer = ""
            pieces.append(token)
    if digit_buffer:
        pieces.append(digit_buffer)
    return " ".join(pieces)


def split_single_row_plate_text(text):
    tokens = normalize_text(text).split() if "normalize_text" in globals() else text.split()
    digit_start = len(tokens)
    for index, token in enumerate(tokens):
        if all(char in BANGLA_DIGIT_VALUES for char in token):
            digit_start = index
            break
    top_text = " ".join(tokens[:digit_start]).strip()
    bottom_text = " ".join(tokens[digit_start:]).strip()
    return top_text, bottom_text


def plate_rows_from_label(label_path):
    rows = split_token_rows(read_character_tokens(label_path))
    return [join_row_tokens([item["token"] for item in row]) for row in rows]


def plate_text_from_label(label_path):
    return " ".join(plate_rows_from_label(label_path)).strip()


CITY_TOKENS = [TARGET_TOKEN_MAP["dhaka"], TARGET_TOKEN_MAP["chattro"], TARGET_TOKEN_MAP["sylhet"]]
METRO_TOKEN = TARGET_TOKEN_MAP["metro"]
SERIES_TOKEN_KEYS = ["ka", "kha", "ga", "gha", "cha", "chha", "ja", "jha", "Ta", "Tha", "ba", "bha", "m", "na", "sa"]
SERIES_TOKENS = {TARGET_TOKEN_MAP[key] for key in SERIES_TOKEN_KEYS}


def token_is_digit_group(token):
    return bool(token) and all(char in BANGLA_DIGIT_VALUES for char in token)


def clean_plate_fields_from_text(text):
    tokens = [token for token in str(text).split() if token]
    city = next((token for token in tokens if token in CITY_TOKENS), "")
    metro = METRO_TOKEN if METRO_TOKEN in tokens else ""
    series_candidates = [token for token in tokens if token in SERIES_TOKENS]
    series = series_candidates[-1] if series_candidates else ""
    digits = "".join(char for token in tokens for char in token if char in BANGLA_DIGIT_VALUES)

    top_parts = [part for part in [city, metro, series] if part]
    top_text = " ".join(top_parts).strip()
    bottom_text = digits.strip()
    return top_text, bottom_text


def split_plate_rows_for_ocr(row_texts):
    row_texts = [text.strip() for text in row_texts if str(text).strip()]
    if not row_texts:
        return "", "", 0

    raw_text = " ".join(row_texts).strip()
    clean_top, clean_bottom = clean_plate_fields_from_text(raw_text)
    if clean_top and clean_bottom:
        return clean_top, clean_bottom, len(row_texts)

    if len(row_texts) == 1:
        top_text, bottom_text = split_single_row_plate_text(row_texts[0])
        return top_text, bottom_text, 1
    top_text = " ".join(row_texts[:-1]).strip()
    bottom_text = row_texts[-1].strip()
    return top_text, bottom_text, len(row_texts)



In [ ]:
def build_ocr_manifest(split):
    deblur_manifest_path = PAIR_ROOT / split / "manifest.csv"
    require_path(deblur_manifest_path, f"DeblurGAN {split} manifest")
    deblur_frame = pd.read_csv(deblur_manifest_path)
    records = []

    for row in deblur_frame.itertuples(index=False):
        row_dict = row._asdict()
        source_image = project_path(row_dict["source_image"])
        blurry_crop = project_path(row_dict["blurry"])
        sharp_crop = project_path(row_dict["sharp"])
        deblurred_crop = DEBLURRED_OCR_INPUT_ROOT / split / f"{blurry_crop.stem}_deblurred.png"

        records.append({
            "split": split,
            "image_id": source_image.stem,
            "variant": row_dict.get("variant", 0),
            "target_text": "",
            "raw_target_text": "",
            "top_text": "",
            "bottom_text": "",
            "row_count": 2,
            "source_image": relative_text(source_image),
            "source_label": relative_text(project_path(row_dict.get("source_label", ""))) if row_dict.get("source_label") else "",
            "blurry_crop": relative_text(blurry_crop),
            "sharp_crop": relative_text(sharp_crop),
            "ocr_image": relative_text(deblurred_crop),
            "x1": row_dict.get("x1"),
            "y1": row_dict.get("y1"),
            "x2": row_dict.get("x2"),
            "y2": row_dict.get("y2"),
        })

    frame = pd.DataFrame(records)
    frame.to_csv(MANIFEST_ROOT / f"{split}_ocr_manifest.csv", index=False, encoding="utf-8-sig")
    return frame


ocr_manifests = {split: build_ocr_manifest(split) for split in SPLITS}
for split, frame in ocr_manifests.items():
    print(f"{split}: {len(frame)} clean-dataset OCR input samples")

print("This clean dataset has no OCR text ground truth, so this notebook will create OCR inputs and predictions only.")
display(pd.concat(ocr_manifests.values(), ignore_index=True).head(10)[["split", "image_id", "variant", "blurry_crop", "ocr_image"]])


In [ ]:
def show_target_examples(frame, count=6):
    sample = frame.sample(min(count, len(frame)), random_state=7) if len(frame) else frame
    if sample.empty:
        print("No samples to show.")
        return
    figure, axes = plt.subplots(len(sample), 2, figsize=(10, 2.8 * len(sample)))
    if len(sample) == 1:
        axes = np.array([axes])
    for axis_row, row in zip(axes, sample.itertuples(index=False)):
        image = cv2.imread(str(project_path(row.blurry_crop)))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        axis_row[0].imshow(image)
        axis_row[0].set_title("DeblurGAN blurry input")
        axis_row[0].axis("off")
        axis_row[1].text(0.02, 0.6, row.target_text, fontsize=18)
        axis_row[1].set_title("Reconstructed OCR target")
        axis_row[1].axis("off")
    plt.tight_layout()
    plt.show()


show_target_examples(ocr_manifests["test"])



## 2. Create DeblurGAN OCR Inputs

This section loads the DeblurGAN generator definition from `deblurgan_workflow.ipynb`, loads `runs/deblurgan_clean_dataset_irv2_fpn_psnr_v2/checkpoints/best_generator.pt`, and restores the PSNR-v2 DeblurGAN blurry plate crops. The structured OCR section later compares OCR on `blurry_crop`, DeblurGAN `ocr_image`, and paired `sharp_crop` inputs before choosing the best deployable variant.


In [ ]:
def load_deblurgan_generator_definition():
    notebook = json.loads(DEBLUR_NOTEBOOK.read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        source = "".join(cell.get("source", []))
        if "class FPNInceptionResNetV2Generator" in source:
            end = source.find("class PatchDiscriminator")
            if end == -1:
                raise RuntimeError("Could not isolate DeblurGAN generator definition.")
            generator_source = source[:end]
            break
    else:
        raise RuntimeError("Could not find DeblurGAN generator definition.")

    # The saved checkpoint in this project uses the notebook-local backbone.
    globals()["timm"] = None
    exec(generator_source, globals())


load_deblurgan_generator_definition()
print("Loaded FPNInceptionResNetV2Generator from deblurgan_workflow.ipynb")



In [ ]:
def load_deblur_generator(checkpoint_path=DEBLUR_CHECKPOINT):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    checkpoint = torch.load(checkpoint_path, map_location=DEBLUR_DEVICE)
    state_dict = checkpoint["generator"] if isinstance(checkpoint, dict) and "generator" in checkpoint else checkpoint
    generator = FPNInceptionResNetV2Generator(pretrained_backbone=False).to(DEBLUR_DEVICE)
    generator.load_state_dict(state_dict)
    generator.eval()
    config = checkpoint.get("config", {}) if isinstance(checkpoint, dict) else {}
    return generator, config


def crop_to_tensor(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    image = cv2.resize(image, (PLATE_WIDTH, PLATE_HEIGHT), interpolation=cv2.INTER_AREA)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return torch.from_numpy(image.transpose(2, 0, 1))


def tensor_to_bgr_image(tensor):
    image = tensor.detach().cpu().clamp(0, 1).numpy().transpose(1, 2, 0)
    image = (image * 255.0).round().astype(np.uint8)
    return cv2.cvtColor(image, cv2.COLOR_RGB2BGR)


def is_cuda_oom(error):
    return isinstance(error, RuntimeError) and "out of memory" in str(error).lower()


def write_deblurred_batch(generator, batch_rows, device):
    batch = torch.stack([crop_to_tensor(project_path(row.blurry_crop)) for row in batch_rows]).to(device, non_blocking=True)
    restored = generator(batch)
    for item, output_tensor in zip(batch_rows, restored):
        output_path = project_path(item.ocr_image)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(output_path), tensor_to_bgr_image(output_tensor))
    del batch, restored


def create_deblurred_ocr_inputs(manifests, overwrite=False, batch_size=1):
    global DEBLUR_DEVICE
    generator, config = load_deblur_generator()
    effective_batch_size = max(1, int(batch_size))
    print("Loaded DeblurGAN generator config:", config)
    print("Deblur inference device:", DEBLUR_DEVICE)

    with torch.inference_mode():
        for split, frame in manifests.items():
            output_dir = DEBLURRED_OCR_INPUT_ROOT / split
            output_dir.mkdir(parents=True, exist_ok=True)
            rows = list(frame.itertuples(index=False))
            pending = [row for row in rows if overwrite or not project_path(row.ocr_image).exists()]
            print(f"{split}: {len(rows)} manifest rows, {len(pending)} deblur outputs to create")

            start = 0
            while start < len(pending):
                batch_rows = pending[start:start + effective_batch_size]
                try:
                    write_deblurred_batch(generator, batch_rows, DEBLUR_DEVICE)
                    start += len(batch_rows)
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                except RuntimeError as exc:
                    if is_cuda_oom(exc) and DEBLUR_DEVICE.type == "cuda":
                        torch.cuda.empty_cache()
                        if effective_batch_size > 1:
                            effective_batch_size = max(1, effective_batch_size // 2)
                            print(f"CUDA OOM. Reducing DeblurGAN OCR batch size to {effective_batch_size} and retrying.")
                            continue
                        print("CUDA OOM at batch size 1. Switching DeblurGAN OCR input creation to CPU.")
                        DEBLUR_DEVICE = torch.device("cpu")
                        generator = generator.cpu()
                        effective_batch_size = 1
                        continue
                    raise

            frame.to_csv(MANIFEST_ROOT / f"{split}_ocr_manifest.csv", index=False, encoding="utf-8-sig")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return manifests


ocr_manifests = create_deblurred_ocr_inputs(
    ocr_manifests,
    overwrite=OVERWRITE_DEBLURRED_OCR_INPUTS,
    batch_size=DEBLUR_BATCH_SIZE,
)

missing = []
for split, frame in ocr_manifests.items():
    missing.extend([path for path in frame["ocr_image"] if not project_path(path).exists()])
print("Missing deblurred OCR inputs:", len(missing))
assert not missing, "All OCR inputs must be DeblurGAN-restored crops."


In [ ]:
def show_deblurred_ocr_inputs(frame, count=5):
    sample = frame.sample(min(count, len(frame)), random_state=11) if len(frame) else frame
    if sample.empty:
        print("No OCR inputs to show.")
        return
    figure, axes = plt.subplots(len(sample), 3, figsize=(12, 2.8 * len(sample)))
    if len(sample) == 1:
        axes = np.array([axes])
    for axis_row, row in zip(axes, sample.itertuples(index=False)):
        for axis, path, title in zip(
            axis_row,
            [row.blurry_crop, row.ocr_image, row.sharp_crop],
            ["DeblurGAN blurry input", "DeblurGAN OCR input", "Clean target crop"],
        ):
            image = cv2.imread(str(project_path(path)))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            axis.imshow(image)
            axis.set_title(title)
            axis.axis("off")
    plt.tight_layout()
    plt.show()


show_deblurred_ocr_inputs(ocr_manifests["test"])



## 3. OCR Evaluation Helpers

CER is normalized character edit distance. WER is normalized token edit distance after whitespace normalization. Exact plate match accuracy is the fraction of samples where the normalized prediction exactly equals the normalized target.



In [ ]:
ASCII_TO_BANGLA_DIGIT = str.maketrans(BANGLA_DIGITS)
PUNCTUATION_PATTERN = re.compile(r"[-_:,.;/\|()\[\]{}]+")


def normalize_text(text):
    if text is None or (isinstance(text, float) and math.isnan(text)):
        text = ""
    text = unicodedata.normalize("NFC", str(text))
    text = text.translate(ASCII_TO_BANGLA_DIGIT)
    text = text.replace("\n", " ").replace("\r", " ")
    text = PUNCTUATION_PATTERN.sub(" ", text)
    kept = []
    for char in text:
        if char.isspace():
            kept.append(" ")
        elif "\u0980" <= char <= "\u09ff":
            kept.append(char)
        elif char.isalnum():
            kept.append(char.lower())
    return " ".join("".join(kept).split())


def edit_distance(reference, hypothesis):
    reference = list(reference)
    hypothesis = list(hypothesis)
    previous = list(range(len(hypothesis) + 1))
    for i, ref_item in enumerate(reference, start=1):
        current = [i]
        for j, hyp_item in enumerate(hypothesis, start=1):
            cost = 0 if ref_item == hyp_item else 1
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + cost))
        previous = current
    return previous[-1]


def character_error_rate(reference, hypothesis):
    reference = normalize_text(reference)
    hypothesis = normalize_text(hypothesis)
    return edit_distance(reference, hypothesis) / max(1, len(reference))


def word_error_rate(reference, hypothesis):
    reference_words = normalize_text(reference).split()
    hypothesis_words = normalize_text(hypothesis).split()
    return edit_distance(reference_words, hypothesis_words) / max(1, len(reference_words))


def evaluate_predictions(predictions, method_name):
    records = []
    for row in predictions.itertuples(index=False):
        reference = normalize_text(getattr(row, "target_text", ""))
        prediction = normalize_text(getattr(row, "prediction", ""))
        has_reference = bool(reference) and HAS_OCR_GROUND_TRUTH
        records.append({
            "method": method_name,
            "split": row.split,
            "image_id": row.image_id,
            "target_text": getattr(row, "target_text", ""),
            "prediction": getattr(row, "prediction", ""),
            "normalized_target": reference,
            "normalized_prediction": prediction,
            "cer": character_error_rate(reference, prediction) if has_reference else np.nan,
            "wer": word_error_rate(reference, prediction) if has_reference else np.nan,
            "exact_match": int(reference == prediction) if has_reference else np.nan,
            "ocr_image": row.ocr_image,
        })
    return pd.DataFrame(records)


def empty_predictions(frame, message):
    result = frame[["split", "image_id", "target_text", "ocr_image"]].copy()
    result["prediction"] = ""
    result["note"] = message
    return result


## 4. Main OCR: EasyOCR

EasyOCR runs on the DeblurGAN output crops only. The first run may download EasyOCR model weights in your notebook environment.



In [ ]:
def crop_ocr_row_image(image, field, row_count=2):
    height, width = image.shape[:2]
    if row_count <= 1:
        return image
    if field == "top":
        y1, y2 = 0, int(height * 0.66)
    elif field == "bottom":
        y1, y2 = int(height * 0.40), height
    else:
        y1, y2 = 0, height
    return image[max(0, y1):min(height, y2), :]


def run_easyocr(frame):
    try:
        import easyocr
    except Exception as exc:
        print("EasyOCR is not available:", exc)
        return empty_predictions(frame, "easyocr import failed")

    reader = easyocr.Reader(EASYOCR_LANGS, gpu=torch.cuda.is_available())
    records = []
    for row in tqdm(list(frame.itertuples(index=False)), desc="EasyOCR row-wise test"):
        image_path = project_path(row.ocr_image)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            prediction = ""
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            parts = []
            for field in ["top", "bottom"]:
                row_image = crop_ocr_row_image(image, field, getattr(row, "row_count", 2))
                try:
                    result = reader.readtext(row_image, detail=0, paragraph=False, decoder="beamsearch")
                    parts.append(" ".join(str(part) for part in result))
                except Exception as exc:
                    parts.append("")
                    print(f"EasyOCR failed on {image_path.name} {field}: {exc}")
            prediction = " ".join(part for part in parts if part).strip()
        records.append({
            "split": row.split,
            "image_id": row.image_id,
            "target_text": row.target_text,
            "ocr_image": row.ocr_image,
            "prediction": prediction,
        })
    predictions = pd.DataFrame(records)
    predictions.to_csv(PREDICTION_ROOT / "easyocr_predictions.csv", index=False, encoding="utf-8-sig")
    return predictions


easyocr_predictions = run_easyocr(ocr_manifests["test"])
easyocr_eval = evaluate_predictions(easyocr_predictions, "EasyOCR row-wise")
easyocr_eval.to_csv(PREDICTION_ROOT / "easyocr_evaluation.csv", index=False, encoding="utf-8-sig")
display(easyocr_eval.head())



## 5. Baseline OCR: Tesseract Bengali

Tesseract is the baseline comparison. It uses the same DeblurGAN-restored crop images with a simple grayscale, resize, and threshold preprocessing step.



In [ ]:
def preprocess_for_tesseract_array(image, scale=4):
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    image = cv2.GaussianBlur(image, (3, 3), 0)
    image = cv2.equalizeHist(image)
    _, image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return image


def run_tesseract(frame):
    try:
        import pytesseract
    except Exception as exc:
        print("pytesseract is not available:", exc)
        return empty_predictions(frame, "pytesseract import failed")

    records = []
    for row in tqdm(list(frame.itertuples(index=False)), desc="Tesseract row-wise test"):
        image_path = project_path(row.ocr_image)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            prediction = ""
        else:
            parts = []
            for field in ["top", "bottom"]:
                try:
                    row_image = crop_ocr_row_image(image, field, getattr(row, "row_count", 2))
                    processed = preprocess_for_tesseract_array(row_image)
                    parts.append(pytesseract.image_to_string(processed, lang=TESSERACT_LANG, config=TESSERACT_CONFIG))
                except Exception as exc:
                    parts.append("")
                    print(f"Tesseract failed on {image_path.name} {field}: {exc}")
            prediction = " ".join(parts).strip()
        records.append({
            "split": row.split,
            "image_id": row.image_id,
            "target_text": row.target_text,
            "ocr_image": row.ocr_image,
            "prediction": prediction,
        })
    predictions = pd.DataFrame(records)
    predictions.to_csv(PREDICTION_ROOT / "tesseract_predictions.csv", index=False, encoding="utf-8-sig")
    return predictions


tesseract_predictions = run_tesseract(ocr_manifests["test"])
tesseract_eval = evaluate_predictions(tesseract_predictions, "Tesseract Bengali row-wise")
tesseract_eval.to_csv(PREDICTION_ROOT / "tesseract_evaluation.csv", index=False, encoding="utf-8-sig")
display(tesseract_eval.head())



## 6. Structured OCR: Variant Comparison + Valid Plate Decoding

**OCR pipeline:** DeblurGAN PSNR-v2 plate crop -> compare blurry, DeblurGAN-restored, and sharp crop inputs -> split plate into top and bottom rows -> classify top-row city, metro flag, and series/class letter -> recognize the bottom row as exactly six Bangla digits -> apply valid-plate post-processing -> evaluate CER, WER, and exact plate match accuracy.

This section replaces the old top-row CTC recognizer with a structured model. The upper row is decoded as city, metro flag, and series/class labels; the lower row is decoded as exactly six Bangla digits. Training uses class-balanced sampling and optional synthetic Bangla plate images, then evaluates OCR separately on blurry, DeblurGAN-restored, and sharp crop variants.


In [ ]:
def image_variant_path(row, variant):
    if variant not in OCR_IMAGE_VARIANTS:
        raise KeyError(f"Unknown OCR image variant: {variant}")
    column = OCR_IMAGE_VARIANTS[variant]
    if isinstance(row, pd.Series):
        value = row[column]
    elif isinstance(row, dict):
        value = row[column]
    else:
        value = getattr(row, column)
    return project_path(value)


def safe_row_count(value, default=2):
    try:
        if pd.isna(value):
            return default
        return int(value)
    except Exception:
        return default


def read_variant_image(row, variant):
    image_path = image_variant_path(row, variant)
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Could not read {variant} OCR image: {image_path}")
    return image


def enhance_row_image(image):
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image = cv2.resize(image, (OCR_IMAGE_WIDTH, OCR_IMAGE_HEIGHT), interpolation=cv2.INTER_CUBIC)
    image = cv2.equalizeHist(image)
    return image


def augment_row_image(image):
    if not USE_OCR_AUGMENTATION:
        return image
    if np.random.rand() < 0.70:
        alpha = np.random.uniform(0.80, 1.25)
        beta = np.random.uniform(-22, 22)
        image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    if np.random.rand() < 0.45:
        image = cv2.GaussianBlur(image, (3, 3), 0)
    if np.random.rand() < 0.40:
        noise = np.random.normal(0, np.random.uniform(2, 9), image.shape).astype(np.float32)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if np.random.rand() < 0.40:
        height, width = image.shape[:2]
        angle = np.random.uniform(-3.0, 3.0)
        matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, np.random.uniform(0.96, 1.04))
        image = cv2.warpAffine(image, matrix, (width, height), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return image


def field_tensor_from_image(image, field, row_count=2, augment=False):
    row_image = crop_ocr_row_image(image, field, row_count)
    row_image = enhance_row_image(row_image)
    if augment:
        row_image = augment_row_image(row_image)
    row_image = row_image.astype(np.float32) / 255.0
    row_image = (row_image - 0.5) / 0.5
    return torch.from_numpy(row_image).unsqueeze(0)


def keep_bangla_digits(text):
    text = normalize_text(text)
    return "".join(char for char in text if char in BANGLA_DIGIT_VALUES)


BANGLA_DIGIT_LIST = [BANGLA_DIGITS[str(index)] for index in range(10)]
DIGIT_TO_INDEX = {digit: index for index, digit in enumerate(BANGLA_DIGIT_LIST)}
INDEX_TO_DIGIT = {index: digit for digit, index in DIGIT_TO_INDEX.items()}


def parse_top_components(top_text):
    tokens = normalize_text(top_text).split()
    city = next((token for token in tokens if token in CITY_TOKENS), "")
    metro_flag = int(METRO_TOKEN in tokens)
    series_candidates = [token for token in tokens if token in SERIES_TOKENS]
    series = series_candidates[-1] if series_candidates else ""
    return city, metro_flag, series


def compose_top_text(city, metro_flag, series):
    parts = []
    if city:
        parts.append(city)
    if int(metro_flag):
        parts.append(METRO_TOKEN)
    if series:
        parts.append(series)
    return normalize_text(" ".join(parts))


def ordered_vocab(values):
    cleaned = sorted({normalize_text(value) for value in values if normalize_text(value)})
    return [""] + cleaned


def collect_top_components(frames):
    cities, series_values, top_texts = set(), set(), set()
    for frame in frames:
        for top_text in frame["top_text"].fillna("").tolist():
            top_text = normalize_text(top_text)
            city, _, series = parse_top_components(top_text)
            cities.add(city)
            series_values.add(series)
            if top_text:
                top_texts.add(top_text)
    return cities, series_values, top_texts


TOP_COMPONENT_SOURCE_FRAMES = [ocr_manifests["train"], ocr_manifests["valid"]]
TOP_CITY_VALUES, TOP_SERIES_VALUES, TOP_TEXT_VALUES = collect_top_components(TOP_COMPONENT_SOURCE_FRAMES)
TOP_CITY_VOCAB = ordered_vocab(TOP_CITY_VALUES)
TOP_SERIES_VOCAB = ordered_vocab(TOP_SERIES_VALUES)
TOP_TEXT_VOCAB = sorted(TOP_TEXT_VALUES)
BOTTOM_TEXT_VOCAB = sorted({keep_bangla_digits(text) for frame in TOP_COMPONENT_SOURCE_FRAMES for text in frame["bottom_text"].fillna("").tolist() if keep_bangla_digits(text)})

TOP_CITY_TO_INDEX = {value: index for index, value in enumerate(TOP_CITY_VOCAB)}
TOP_SERIES_TO_INDEX = {value: index for index, value in enumerate(TOP_SERIES_VOCAB)}

def printable_vocab(values):
    return " | ".join(values).encode("unicode_escape").decode("ascii")


print("Top city vocab:", printable_vocab(TOP_CITY_VOCAB))
print("Top series/class vocab:", printable_vocab(TOP_SERIES_VOCAB))
print("Valid top-row combinations:", len(TOP_TEXT_VOCAB))
print("Known bottom numbers:", len(BOTTOM_TEXT_VOCAB))


def nearest_vocab_text(text, vocabulary, max_rate=0.45):
    text = normalize_text(text)
    if not text or not vocabulary:
        return text
    best = min(vocabulary, key=lambda candidate: character_error_rate(candidate, text))
    rate = character_error_rate(best, text)
    return best if rate <= max_rate else text


def postprocess_top_text(text):
    text = normalize_text(text)
    if not text:
        return ""
    return nearest_vocab_text(text, TOP_TEXT_VOCAB, max_rate=0.45)


def enforce_fixed_digit_count(text):
    digits = keep_bangla_digits(text)
    if len(digits) == BOTTOM_DIGIT_COUNT:
        return digits
    if BOTTOM_TEXT_VOCAB:
        best = min(BOTTOM_TEXT_VOCAB, key=lambda candidate: edit_distance(candidate, digits))
        if edit_distance(best, digits) <= max(2, BOTTOM_DIGIT_COUNT // 2):
            return best
    if len(digits) > BOTTOM_DIGIT_COUNT:
        return digits[:BOTTOM_DIGIT_COUNT]
    return digits.ljust(BOTTOM_DIGIT_COUNT, BANGLA_DIGITS["0"])


def compose_valid_plate_prediction(top_prediction, bottom_prediction):
    top_prediction = postprocess_top_text(top_prediction)
    bottom_prediction = enforce_fixed_digit_count(bottom_prediction)
    return " ".join(part for part in [top_prediction, bottom_prediction] if part).strip()


def find_bangla_font_path():
    if ImageFont is None:
        return None
    candidates = [
        PROJECT_ROOT / "fonts" / "Nirmala.ttf",
        PROJECT_ROOT / "fonts" / "Vrinda.ttf",
        Path("C:/Windows/Fonts/Nirmala.ttf"),
        Path("C:/Windows/Fonts/NirmalaB.ttf"),
        Path("C:/Windows/Fonts/vrinda.ttf"),
        Path("C:/Windows/Fonts/Vrinda.ttf"),
        Path("C:/Windows/Fonts/kalpurush.ttf"),
        Path("C:/Windows/Fonts/Siyamrupali.ttf"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def text_bbox(draw, xy, text, font):
    try:
        return draw.textbbox(xy, text, font=font)
    except Exception:
        width, height = draw.textsize(text, font=font)
        x, y = xy
        return (x, y, x + width, y + height)


def draw_centered_text(draw, text, font, center_y, fill=(12, 35, 28)):
    if not text:
        return
    bbox = text_bbox(draw, (0, 0), text, font)
    width = bbox[2] - bbox[0]
    height = bbox[3] - bbox[1]
    x = max(4, (PLATE_WIDTH - width) // 2)
    y = int(center_y - height / 2)
    draw.text((x, y), text, font=font, fill=fill)


def perturb_synthetic_plate(image):
    if np.random.rand() < 0.55:
        source = np.float32([[0, 0], [PLATE_WIDTH - 1, 0], [PLATE_WIDTH - 1, PLATE_HEIGHT - 1], [0, PLATE_HEIGHT - 1]])
        jitter = np.random.uniform(-5, 5, source.shape).astype(np.float32)
        target = source + jitter
        matrix = cv2.getPerspectiveTransform(source, target)
        image = cv2.warpPerspective(image, matrix, (PLATE_WIDTH, PLATE_HEIGHT), borderMode=cv2.BORDER_REPLICATE)
    if np.random.rand() < 0.65:
        image = cv2.GaussianBlur(image, (3, 3), 0)
    if np.random.rand() < 0.70:
        alpha = np.random.uniform(0.75, 1.20)
        beta = np.random.uniform(-18, 18)
        image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    if np.random.rand() < 0.50:
        noise = np.random.normal(0, np.random.uniform(2, 8), image.shape).astype(np.float32)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return image


def make_synthetic_plate_image(top_text, bottom_text, font_path):
    canvas = Image.new("RGB", (PLATE_WIDTH, PLATE_HEIGHT), (235, 238, 228))
    draw = ImageDraw.Draw(canvas)
    draw.rectangle((2, 2, PLATE_WIDTH - 3, PLATE_HEIGHT - 3), outline=(20, 60, 45), width=2)
    top_font = ImageFont.truetype(str(font_path), 25)
    bottom_font = ImageFont.truetype(str(font_path), 34)
    draw_centered_text(draw, normalize_text(top_text), top_font, 30)
    draw_centered_text(draw, normalize_text(bottom_text), bottom_font, 68)
    image = cv2.cvtColor(np.array(canvas), cv2.COLOR_RGB2BGR)
    return perturb_synthetic_plate(image)


def random_bangla_digits(count=BOTTOM_DIGIT_COUNT):
    return "".join(random.choice(BANGLA_DIGIT_LIST) for _ in range(count))


def build_synthetic_plate_samples(train_frame):
    if not USE_SYNTHETIC_OCR_DATA:
        return []
    font_path = find_bangla_font_path()
    if font_path is None:
        print("No Bangla-capable font found; synthetic OCR samples are skipped.")
        return []

    top_counts = Counter(normalize_text(value) for value in train_frame["top_text"].fillna("").tolist())
    top_candidates = sorted(value for value in top_counts if value)
    if not top_candidates:
        return []

    samples = []
    for top_text in top_candidates:
        # Rare top labels get more synthetic examples, which directly fights the common-label collapse.
        rarity_boost = max(1, int(np.ceil(max(top_counts.values()) / max(1, top_counts[top_text]))))
        rarity_boost = min(SYNTHETIC_RARITY_BOOST_CAP, rarity_boost)
        class_count = SYNTHETIC_SAMPLES_PER_TOP_CLASS * rarity_boost
        for sample_index in range(class_count):
            bottom_text = random_bangla_digits()
            samples.append({
                "image": make_synthetic_plate_image(top_text, bottom_text, font_path),
                "top_text": top_text,
                "bottom_text": bottom_text,
                "row_count": 2,
                "image_id": f"synthetic_{len(samples):06d}",
            })

    for _ in range(SYNTHETIC_RANDOM_SAMPLES):
        top_text = random.choice(top_candidates)
        bottom_text = random_bangla_digits()
        samples.append({
            "image": make_synthetic_plate_image(top_text, bottom_text, font_path),
            "top_text": top_text,
            "bottom_text": bottom_text,
            "row_count": 2,
            "image_id": f"synthetic_{len(samples):06d}",
        })

    print(f"Synthetic OCR samples: {len(samples)} using font {font_path}")
    return samples


def make_balanced_sampler(labels):
    if not USE_CLASS_BALANCING or not labels:
        return None
    counts = Counter(labels)
    weights = torch.DoubleTensor([1.0 / counts[label] for label in labels])
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


class StructuredTopDataset(Dataset):
    def __init__(self, frame, variants, synthetic_samples=None, augment=False):
        self.samples = []
        self.augment = augment
        for _, row in frame.iterrows():
            for variant in variants:
                image_path = image_variant_path(row, variant)
                if image_path.exists():
                    self.samples.append({
                        "kind": "real",
                        "variant": variant,
                        "image_path": image_path,
                        "top_text": normalize_text(row["top_text"]),
                        "bottom_text": normalize_text(row["bottom_text"]),
                        "row_count": safe_row_count(row.get("row_count", 2)),
                        "image_id": row["image_id"],
                    })
        for sample in synthetic_samples or []:
            self.samples.append({"kind": "synthetic", **sample})
        self.labels = [normalize_text(sample["top_text"]) for sample in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = sample["image"].copy() if sample["kind"] == "synthetic" else cv2.imread(str(sample["image_path"]), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Could not read OCR top sample: {sample.get('image_path')}")
        city, metro_flag, series = parse_top_components(sample["top_text"])
        return {
            "image": field_tensor_from_image(image, "top", sample.get("row_count", 2), augment=self.augment),
            "city_label": TOP_CITY_TO_INDEX.get(city, 0),
            "metro_label": int(metro_flag),
            "series_label": TOP_SERIES_TO_INDEX.get(series, 0),
            "top_text": compose_top_text(city, metro_flag, series),
            "image_id": sample["image_id"],
        }

    def sample_labels(self):
        return self.labels


class FixedDigitDataset(Dataset):
    def __init__(self, frame, variants, synthetic_samples=None, augment=False):
        self.samples = []
        self.augment = augment
        for _, row in frame.iterrows():
            bottom_text = keep_bangla_digits(row["bottom_text"])
            if len(bottom_text) != BOTTOM_DIGIT_COUNT:
                continue
            for variant in variants:
                image_path = image_variant_path(row, variant)
                if image_path.exists():
                    self.samples.append({
                        "kind": "real",
                        "variant": variant,
                        "image_path": image_path,
                        "bottom_text": bottom_text,
                        "row_count": safe_row_count(row.get("row_count", 2)),
                        "image_id": row["image_id"],
                    })
        for sample in synthetic_samples or []:
            bottom_text = keep_bangla_digits(sample["bottom_text"])
            if len(bottom_text) == BOTTOM_DIGIT_COUNT:
                self.samples.append({"kind": "synthetic", **sample, "bottom_text": bottom_text})
        self.labels = [sample["bottom_text"] for sample in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = sample["image"].copy() if sample["kind"] == "synthetic" else cv2.imread(str(sample["image_path"]), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Could not read OCR bottom sample: {sample.get('image_path')}")
        target = torch.tensor([DIGIT_TO_INDEX[digit] for digit in sample["bottom_text"]], dtype=torch.long)
        return {
            "image": field_tensor_from_image(image, "bottom", sample.get("row_count", 2), augment=self.augment),
            "target": target,
            "bottom_text": sample["bottom_text"],
            "image_id": sample["image_id"],
        }

    def sample_labels(self):
        return self.labels


def top_collate(batch):
    return {
        "images": torch.stack([item["image"] for item in batch]),
        "city_labels": torch.tensor([item["city_label"] for item in batch], dtype=torch.long),
        "metro_labels": torch.tensor([item["metro_label"] for item in batch], dtype=torch.long),
        "series_labels": torch.tensor([item["series_label"] for item in batch], dtype=torch.long),
        "top_texts": [item["top_text"] for item in batch],
        "image_ids": [item["image_id"] for item in batch],
    }


def digit_collate(batch):
    return {
        "images": torch.stack([item["image"] for item in batch]),
        "targets": torch.stack([item["target"] for item in batch]),
        "bottom_texts": [item["bottom_text"] for item in batch],
        "image_ids": [item["image_id"] for item in batch],
    }


def make_top_loader(frame, variants, synthetic_samples=None, shuffle=False, augment=False):
    dataset = StructuredTopDataset(frame, variants, synthetic_samples=synthetic_samples, augment=augment)
    sampler = make_balanced_sampler(dataset.sample_labels()) if shuffle else None
    return DataLoader(
        dataset,
        batch_size=STRUCTURED_BATCH_SIZE,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=0,
        collate_fn=top_collate,
    )


def make_digit_loader(frame, variants, synthetic_samples=None, shuffle=False, augment=False):
    dataset = FixedDigitDataset(frame, variants, synthetic_samples=synthetic_samples, augment=augment)
    sampler = make_balanced_sampler(dataset.sample_labels()) if shuffle else None
    return DataLoader(
        dataset,
        batch_size=STRUCTURED_BATCH_SIZE,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=0,
        collate_fn=digit_collate,
    )


class OCRFeatureBackbone(nn.Module):
    def __init__(self, channels=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 48, kernel_size=3, padding=1), nn.BatchNorm2d(48), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(48, 96, kernel_size=3, padding=1), nn.BatchNorm2d(96), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(96, 192, kernel_size=3, padding=1), nn.BatchNorm2d(192), nn.ReLU(inplace=True),
            nn.Conv2d(192, channels, kernel_size=3, padding=1), nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
        )
        self.out_channels = channels

    def forward(self, images):
        return self.features(images)


class TopComponentClassifier(nn.Module):
    def __init__(self, city_classes, series_classes):
        super().__init__()
        self.backbone = OCRFeatureBackbone(channels=256)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.30)
        self.city_head = nn.Linear(self.backbone.out_channels, city_classes)
        self.metro_head = nn.Linear(self.backbone.out_channels, 2)
        self.series_head = nn.Linear(self.backbone.out_channels, series_classes)

    def forward(self, images):
        features = self.pool(self.backbone(images)).flatten(1)
        features = self.dropout(features)
        return {
            "city": self.city_head(features),
            "metro": self.metro_head(features),
            "series": self.series_head(features),
        }


class FixedSixDigitRecognizer(nn.Module):
    def __init__(self, digit_classes=10):
        super().__init__()
        self.backbone = OCRFeatureBackbone(channels=256)
        self.sequence_pool = nn.AdaptiveAvgPool2d((1, BOTTOM_DIGIT_COUNT))
        self.sequence = nn.LSTM(self.backbone.out_channels, 192, num_layers=2, bidirectional=True, dropout=0.25, batch_first=True)
        self.classifier = nn.Linear(384, digit_classes)

    def forward(self, images):
        features = self.backbone(images)
        features = self.sequence_pool(features).squeeze(2).permute(0, 2, 1)
        sequence, _ = self.sequence(features)
        return self.classifier(sequence)


def top_epoch_loss(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    city_loss_fn = nn.CrossEntropyLoss()
    metro_loss_fn = nn.CrossEntropyLoss()
    series_loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_items = 0
    correct_exact = 0
    for batch in tqdm(loader, leave=False):
        images = batch["images"].to(DEVICE)
        city_labels = batch["city_labels"].to(DEVICE)
        metro_labels = batch["metro_labels"].to(DEVICE)
        series_labels = batch["series_labels"].to(DEVICE)
        outputs = model(images)
        loss = (
            city_loss_fn(outputs["city"], city_labels)
            + TOP_METRO_LOSS_WEIGHT * metro_loss_fn(outputs["metro"], metro_labels)
            + TOP_CLASS_LOSS_WEIGHT * series_loss_fn(outputs["series"], series_labels)
        )
        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        total_loss += float(loss.item()) * images.size(0)
        total_items += images.size(0)
        predictions = decode_top_outputs(outputs)
        correct_exact += sum(int(prediction == target) for prediction, target in zip(predictions, batch["top_texts"]))
    return {"loss": total_loss / max(1, total_items), "exact": correct_exact / max(1, total_items)}


def digit_epoch_loss(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_items = 0
    correct_exact = 0
    for batch in tqdm(loader, leave=False):
        images = batch["images"].to(DEVICE)
        targets = batch["targets"].to(DEVICE)
        logits = model(images)
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        total_loss += float(loss.item()) * images.size(0)
        total_items += images.size(0)
        predictions = decode_digit_logits(logits)
        correct_exact += sum(int(prediction == target) for prediction, target in zip(predictions, batch["bottom_texts"]))
    return {"loss": total_loss / max(1, total_items), "exact": correct_exact / max(1, total_items)}


def decode_top_outputs(outputs):
    city_indices = outputs["city"].argmax(dim=1).detach().cpu().tolist()
    metro_indices = outputs["metro"].argmax(dim=1).detach().cpu().tolist()
    series_indices = outputs["series"].argmax(dim=1).detach().cpu().tolist()
    predictions = []
    for city_index, metro_index, series_index in zip(city_indices, metro_indices, series_indices):
        city = TOP_CITY_VOCAB[int(city_index)]
        series = TOP_SERIES_VOCAB[int(series_index)]
        predictions.append(compose_top_text(city, int(metro_index), series))
    return predictions


def decode_digit_logits(logits):
    indices = logits.argmax(dim=2).detach().cpu().numpy()
    predictions = []
    for row in indices:
        predictions.append("".join(INDEX_TO_DIGIT[int(index)] for index in row[:BOTTOM_DIGIT_COUNT]))
    return predictions


def save_top_checkpoint(model):
    torch.save({
        "model": model.state_dict(),
        "city_vocab": TOP_CITY_VOCAB,
        "series_vocab": TOP_SERIES_VOCAB,
        "image_height": OCR_IMAGE_HEIGHT,
        "image_width": OCR_IMAGE_WIDTH,
    }, TOP_STRUCTURED_CHECKPOINT)


def save_digit_checkpoint(model):
    torch.save({
        "model": model.state_dict(),
        "digits": BANGLA_DIGIT_LIST,
        "digit_count": BOTTOM_DIGIT_COUNT,
        "image_height": OCR_IMAGE_HEIGHT,
        "image_width": OCR_IMAGE_WIDTH,
    }, BOTTOM_FIXED_DIGIT_CHECKPOINT)


def load_top_checkpoint():
    checkpoint = torch.load(TOP_STRUCTURED_CHECKPOINT, map_location=DEVICE)
    if checkpoint.get("city_vocab") != TOP_CITY_VOCAB or checkpoint.get("series_vocab") != TOP_SERIES_VOCAB:
        raise ValueError("Saved top classifier vocab does not match the current training manifest.")
    model = TopComponentClassifier(len(TOP_CITY_VOCAB), len(TOP_SERIES_VOCAB)).to(DEVICE)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return model


def load_digit_checkpoint():
    checkpoint = torch.load(BOTTOM_FIXED_DIGIT_CHECKPOINT, map_location=DEVICE)
    if checkpoint.get("digits") != BANGLA_DIGIT_LIST or checkpoint.get("digit_count") != BOTTOM_DIGIT_COUNT:
        raise ValueError("Saved digit recognizer settings do not match the current notebook settings.")
    model = FixedSixDigitRecognizer(digit_classes=len(BANGLA_DIGIT_LIST)).to(DEVICE)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return model


def train_or_load_top_classifier(train_frame, valid_frame, synthetic_samples):
    if TOP_STRUCTURED_CHECKPOINT.exists() and not RETRAIN_STRUCTURED_OCR:
        print("Loading existing top component classifier:", TOP_STRUCTURED_CHECKPOINT)
        return load_top_checkpoint()
    if not RUN_STRUCTURED_OCR_TRAINING:
        print("Structured top classifier training is disabled and no checkpoint was loaded.")
        return None

    model = TopComponentClassifier(len(TOP_CITY_VOCAB), len(TOP_SERIES_VOCAB)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=STRUCTURED_LEARNING_RATE, weight_decay=STRUCTURED_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=6)
    train_loader = make_top_loader(train_frame, DEPLOYABLE_OCR_VARIANTS, synthetic_samples=synthetic_samples, shuffle=True, augment=True)
    valid_loader = make_top_loader(valid_frame, DEPLOYABLE_OCR_VARIANTS, synthetic_samples=None, shuffle=False, augment=False)
    history = []
    best_valid_loss = float("inf")
    patience = 16
    stale_epochs = 0

    for epoch in range(1, STRUCTURED_EPOCHS + 1):
        train_metrics = top_epoch_loss(model, train_loader, optimizer=optimizer)
        with torch.no_grad():
            valid_metrics = top_epoch_loss(model, valid_loader, optimizer=None)
        scheduler.step(valid_metrics["loss"])
        history.append({
            "model": "top_component_classifier",
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_exact": train_metrics["exact"],
            "valid_loss": valid_metrics["loss"],
            "valid_exact": valid_metrics["exact"],
            "lr": optimizer.param_groups[0]["lr"],
        })
        print(f"top classifier epoch {epoch:02d}/{STRUCTURED_EPOCHS} - train {train_metrics['loss']:.4f}/{train_metrics['exact']:.3f} - valid {valid_metrics['loss']:.4f}/{valid_metrics['exact']:.3f}")
        if valid_metrics["loss"] < best_valid_loss:
            best_valid_loss = valid_metrics["loss"]
            stale_epochs = 0
            save_top_checkpoint(model)
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"Stopping top classifier early after {patience} stale epochs.")
                break

    pd.DataFrame(history).to_csv(OCR_RUN_ROOT / "structured_top_classifier_training_history.csv", index=False)
    return load_top_checkpoint()


def train_or_load_digit_recognizer(train_frame, valid_frame, synthetic_samples):
    if BOTTOM_FIXED_DIGIT_CHECKPOINT.exists() and not RETRAIN_STRUCTURED_OCR:
        print("Loading existing fixed digit recognizer:", BOTTOM_FIXED_DIGIT_CHECKPOINT)
        return load_digit_checkpoint()
    if not RUN_STRUCTURED_OCR_TRAINING:
        print("Fixed digit recognizer training is disabled and no checkpoint was loaded.")
        return None

    model = FixedSixDigitRecognizer(digit_classes=len(BANGLA_DIGIT_LIST)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=STRUCTURED_LEARNING_RATE, weight_decay=STRUCTURED_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=6)
    train_loader = make_digit_loader(train_frame, DEPLOYABLE_OCR_VARIANTS, synthetic_samples=synthetic_samples, shuffle=True, augment=True)
    valid_loader = make_digit_loader(valid_frame, DEPLOYABLE_OCR_VARIANTS, synthetic_samples=None, shuffle=False, augment=False)
    history = []
    best_valid_loss = float("inf")
    patience = 16
    stale_epochs = 0

    for epoch in range(1, STRUCTURED_EPOCHS + 1):
        train_metrics = digit_epoch_loss(model, train_loader, optimizer=optimizer)
        with torch.no_grad():
            valid_metrics = digit_epoch_loss(model, valid_loader, optimizer=None)
        scheduler.step(valid_metrics["loss"])
        history.append({
            "model": "fixed_six_digit_recognizer",
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_exact": train_metrics["exact"],
            "valid_loss": valid_metrics["loss"],
            "valid_exact": valid_metrics["exact"],
            "lr": optimizer.param_groups[0]["lr"],
        })
        print(f"digit recognizer epoch {epoch:02d}/{STRUCTURED_EPOCHS} - train {train_metrics['loss']:.4f}/{train_metrics['exact']:.3f} - valid {valid_metrics['loss']:.4f}/{valid_metrics['exact']:.3f}")
        if valid_metrics["loss"] < best_valid_loss:
            best_valid_loss = valid_metrics["loss"]
            stale_epochs = 0
            save_digit_checkpoint(model)
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"Stopping digit recognizer early after {patience} stale epochs.")
                break

    pd.DataFrame(history).to_csv(OCR_RUN_ROOT / "structured_fixed_digit_training_history.csv", index=False)
    return load_digit_checkpoint()


def predict_top_for_variant(model, frame, variant):
    if model is None:
        return [""] * len(frame)
    model.eval()
    predictions = []
    with torch.no_grad():
        for row in tqdm(list(frame.itertuples(index=False)), desc=f"top classifier {variant}"):
            image = read_variant_image(row, variant)
            tensor = field_tensor_from_image(image, "top", getattr(row, "row_count", 2), augment=False).unsqueeze(0).to(DEVICE)
            raw_prediction = decode_top_outputs(model(tensor))[0]
            predictions.append(postprocess_top_text(raw_prediction))
    return predictions


def predict_digits_for_variant(model, frame, variant):
    if model is None:
        return [""] * len(frame)
    model.eval()
    predictions = []
    with torch.no_grad():
        for row in tqdm(list(frame.itertuples(index=False)), desc=f"fixed digits {variant}"):
            image = read_variant_image(row, variant)
            tensor = field_tensor_from_image(image, "bottom", getattr(row, "row_count", 2), augment=False).unsqueeze(0).to(DEVICE)
            raw_prediction = decode_digit_logits(model(tensor))[0]
            predictions.append(enforce_fixed_digit_count(raw_prediction))
    return predictions


def run_structured_ocr(top_model, digit_model, frame, variant):
    top_predictions = predict_top_for_variant(top_model, frame, variant)
    bottom_predictions = predict_digits_for_variant(digit_model, frame, variant)
    records = []
    for row, top_prediction, bottom_prediction in zip(frame.itertuples(index=False), top_predictions, bottom_predictions):
        variant_path = image_variant_path(row, variant)
        records.append({
            "split": row.split,
            "image_id": row.image_id,
            "target_text": row.target_text,
            "top_target": row.top_text,
            "bottom_target": row.bottom_text,
            "top_prediction": top_prediction,
            "bottom_prediction": bottom_prediction,
            "ocr_variant": variant,
            "ocr_image": relative_text(variant_path),
            "prediction": compose_valid_plate_prediction(top_prediction, bottom_prediction),
        })
    return pd.DataFrame(records)


def summarize_structured_variant_evaluations(evaluations):
    return (
        evaluations
        .groupby("ocr_variant", as_index=False)
        .agg(samples=("image_id", "count"), cer=("cer", "mean"), wer=("wer", "mean"), exact_plate_match_accuracy=("exact_match", "mean"))
        .assign(deployable=lambda frame: frame["ocr_variant"].isin(DEPLOYABLE_OCR_VARIANTS))
        .sort_values(["deployable", "exact_plate_match_accuracy", "cer"], ascending=[False, False, True])
    )


def choose_best_deployable_variant(variant_summary):
    deployable = variant_summary[variant_summary["ocr_variant"].isin(DEPLOYABLE_OCR_VARIANTS)].copy()
    if deployable.empty:
        return DEFAULT_DEPLOYABLE_OCR_VARIANT
    deployable = deployable.sort_values(["exact_plate_match_accuracy", "cer", "wer"], ascending=[False, True, True])
    return str(deployable.iloc[0]["ocr_variant"])


def run_structured_variant_comparison(top_model, digit_model, frame):
    prediction_frames = {}
    evaluation_frames = []
    for variant in OCR_EVALUATION_VARIANTS:
        predictions = run_structured_ocr(top_model, digit_model, frame, variant)
        predictions.to_csv(PREDICTION_ROOT / f"structured_ocr_{variant}_predictions.csv", index=False, encoding="utf-8-sig")
        evaluation = evaluate_predictions(predictions[["split", "image_id", "target_text", "ocr_image", "prediction"]], f"Structured OCR ({variant})")
        evaluation["ocr_variant"] = variant
        evaluation["deployable"] = variant in DEPLOYABLE_OCR_VARIANTS
        evaluation.to_csv(PREDICTION_ROOT / f"structured_ocr_{variant}_evaluation.csv", index=False, encoding="utf-8-sig")
        prediction_frames[variant] = predictions
        evaluation_frames.append(evaluation)
    evaluations = pd.concat(evaluation_frames, ignore_index=True)
    evaluations.to_csv(PREDICTION_ROOT / "structured_ocr_variant_evaluations.csv", index=False, encoding="utf-8-sig")
    variant_summary = summarize_structured_variant_evaluations(evaluations)
    variant_summary.to_csv(PREDICTION_ROOT / "structured_ocr_variant_summary.csv", index=False, encoding="utf-8-sig")
    return prediction_frames, evaluations, variant_summary


In [ ]:
if not HAS_OCR_GROUND_TRUTH:
    print("No OCR ground truth is available for clean_dataset_processed. Skipping structured OCR training/evaluation.")
    structured_variant_predictions = {}
    structured_variant_evaluations = pd.DataFrame()
    structured_variant_summary = pd.DataFrame()
    structured_predictions = pd.DataFrame(columns=["split", "image_id", "prediction", "ocr_image"])
    structured_eval = pd.DataFrame(columns=["method", "split", "image_id", "target_text", "prediction", "cer", "wer", "exact_match", "ocr_image"])
else:
    synthetic_train_samples = build_synthetic_plate_samples(ocr_manifests["train"])

    top_component_model = train_or_load_top_classifier(
        ocr_manifests["train"],
        ocr_manifests["valid"],
        synthetic_train_samples,
    )
    fixed_digit_model = train_or_load_digit_recognizer(
        ocr_manifests["train"],
        ocr_manifests["valid"],
        synthetic_train_samples,
    )

    structured_variant_predictions, structured_variant_evaluations, structured_variant_summary = run_structured_variant_comparison(
        top_component_model,
        fixed_digit_model,
        ocr_manifests["test"],
    )

    final_structured_variant = choose_best_deployable_variant(structured_variant_summary)
    print("Final deployable OCR variant:", final_structured_variant)

    structured_predictions = structured_variant_predictions[final_structured_variant].copy()
    structured_predictions.to_csv(PREDICTION_ROOT / "structured_ocr_predictions.csv", index=False, encoding="utf-8-sig")

    structured_eval = structured_variant_evaluations[structured_variant_evaluations["ocr_variant"] == final_structured_variant].copy()
    structured_eval["method"] = f"Structured OCR final ({final_structured_variant})"
    structured_eval.to_csv(PREDICTION_ROOT / "structured_ocr_evaluation.csv", index=False, encoding="utf-8-sig")

    display(structured_variant_summary)
    display(structured_predictions.head())
    display(structured_eval.head())


## 7. Final Metrics and Figures

The summary table is thesis-ready: lower CER/WER is better, and higher exact plate match accuracy is better.



In [ ]:
evaluation_frames = []
for name in ["easyocr_eval", "structured_eval", "tesseract_eval"]:
    frame = globals().get(name)
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        evaluation_frames.append(frame)

if evaluation_frames:
    all_evaluations = pd.concat(evaluation_frames, ignore_index=True)
else:
    all_evaluations = pd.DataFrame(columns=["method", "split", "image_id", "target_text", "prediction", "cer", "wer", "exact_match", "ocr_image"])

all_evaluations.to_csv(PREDICTION_ROOT / "all_ocr_predictions_or_evaluations.csv", index=False, encoding="utf-8-sig")

if HAS_OCR_GROUND_TRUTH and not all_evaluations.empty:
    summary = (
        all_evaluations
        .groupby("method", as_index=False)
        .agg(samples=("image_id", "count"), cer=("cer", "mean"), wer=("wer", "mean"), exact_plate_match_accuracy=("exact_match", "mean"))
        .sort_values(["cer", "wer", "method"])
    )
    summary.to_csv(PREDICTION_ROOT / "ocr_metric_summary.csv", index=False, encoding="utf-8-sig")
    display(summary)
else:
    summary = pd.DataFrame()
    print("Clean dataset OCR ground truth is not available. Saved OCR predictions, but CER/WER/accuracy are skipped.")

if "structured_variant_summary" in globals() and isinstance(structured_variant_summary, pd.DataFrame) and not structured_variant_summary.empty:
    print("Structured OCR input-variant comparison:")
    display(structured_variant_summary)


In [ ]:
def plot_metric_summary(summary_frame):
    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = [
        ("cer", "CER", "lower is better"),
        ("wer", "WER", "lower is better"),
        ("exact_plate_match_accuracy", "Exact Plate Match Accuracy", "higher is better"),
    ]
    colors = ["#3B82F6", "#10B981", "#F59E0B"]
    for axis, (column, title, subtitle), color in zip(axes, metrics, colors):
        axis.bar(summary_frame["method"], summary_frame[column], color=color)
        axis.set_title(f"{title}\n{subtitle}")
        axis.set_ylim(0, 1 if column != "exact_plate_match_accuracy" else max(1, summary_frame[column].max() * 1.15))
        axis.tick_params(axis="x", rotation=25)
    figure.tight_layout()
    output_path = FIGURE_ROOT / "ocr_metric_summary.png"
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    print("Saved:", output_path)
    plt.show()


plot_metric_summary(summary)



In [ ]:
def show_prediction_examples(evaluations, count=8):
    if evaluations.empty:
        print("No evaluations to show.")
        return
    sample_ids = evaluations[["image_id"]].drop_duplicates().sample(min(count, evaluations["image_id"].nunique()), random_state=13)["image_id"].tolist()
    methods = evaluations["method"].drop_duplicates().tolist()
    figure, axes = plt.subplots(len(sample_ids), 2, figsize=(14, 3.2 * len(sample_ids)))
    if len(sample_ids) == 1:
        axes = np.array([axes])
    for axis_row, image_id in zip(axes, sample_ids):
        rows = evaluations[evaluations["image_id"] == image_id]
        first = rows.iloc[0]
        image = cv2.imread(str(project_path(first["ocr_image"])))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        axis_row[0].imshow(image)
        axis_row[0].set_title("DeblurGAN OCR input")
        axis_row[0].axis("off")
        lines = [f"Target: {first['target_text']}"]
        for method in methods:
            match = rows[rows["method"] == method]
            if not match.empty:
                item = match.iloc[0]
                lines.append(f"{method}: {item['prediction']} | CER {item['cer']:.2f} | WER {item['wer']:.2f}")
        axis_row[1].text(0.01, 0.95, "\n".join(lines), va="top", fontsize=11)
        axis_row[1].axis("off")
    figure.tight_layout()
    output_path = FIGURE_ROOT / "ocr_prediction_examples.png"
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    print("Saved:", output_path)
    plt.show()


show_prediction_examples(all_evaluations)



## Saved Outputs

- DeblurGAN OCR inputs: `runs/ocr_clean_dataset_psnr_v2/deblurred_ocr_inputs/`
- OCR manifests for clean-dataset deblurred inputs: `runs/ocr_clean_dataset_psnr_v2/manifests/`
- Structured OCR checkpoints: `runs/ocr_clean_dataset_psnr_v2/checkpoints/structured_ocr/top_component_classifier_best.pt` and `bottom_fixed_digit_best.pt`
- Structured OCR variant comparison: `runs/ocr_clean_dataset_psnr_v2/predictions/structured_ocr_variant_summary.csv`
- Final structured OCR predictions/evaluation: `runs/ocr_clean_dataset_psnr_v2/predictions/structured_ocr_predictions.csv` and `structured_ocr_evaluation.csv`
- Per-model predictions and evaluations: `runs/ocr_clean_dataset_psnr_v2/predictions/`
- OCR metric and qualitative figures: `runs/ocr_clean_dataset_psnr_v2/figures/`
